In [1]:
import pandas as pd

# Load the datasets
train_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/02_cardiovascular_diseases/train.csv'
test_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/02_cardiovascular_diseases/test.csv'

# Read the datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Drop rows with missing values
train_df_clean = train_df.dropna().drop_duplicates()
test_df_clean = test_df.dropna().drop_duplicates()

# Verify the changes
train_df_clean.info()
test_df_clean.info()


<class 'pandas.core.frame.DataFrame'>
Index: 49415 entries, 0 to 49415
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   General_Health                49415 non-null  object 
 1   Checkup                       49415 non-null  object 
 2   Exercise                      49415 non-null  object 
 3   Heart_Disease                 49415 non-null  object 
 4   Skin_Cancer                   49415 non-null  object 
 5   Other_Cancer                  49415 non-null  object 
 6   Depression                    49415 non-null  object 
 7   Diabetes                      49415 non-null  object 
 8   Arthritis                     49415 non-null  object 
 9   Sex                           49415 non-null  object 
 10  Age_Category                  49415 non-null  object 
 11  Height_(cm)                   49415 non-null  float64
 12  Weight_(kg)                   49415 non-null  float64
 13  BMI   

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Get column information for the cleaned training dataset
column_info_train = get_column_info(train_df_clean)
print("Column information for the cleaned training dataset:")
print(column_info_train)

# Get column information for the cleaned test dataset
column_info_test = get_column_info(test_df_clean)
print("Column information for the cleaned test dataset:")
print(column_info_test)


2025-08-30 18:17:43.449 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Column information for the cleaned training dataset:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': []}
Column information for the cleaned test dataset:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': []}


In [3]:
import pandas as pd

# Copy the cleaned datasets to avoid modifying the original data
train_df_copy = train_df_clean.copy()
test_df_copy = test_df_clean.copy()

# Define a function to categorize BMI
def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal weight'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obesity'

# Apply the function to create the new feature 'BMI_Category'
train_df_copy['BMI_Category'] = train_df_copy['BMI'].apply(categorize_bmi).astype('category')
test_df_copy['BMI_Category'] = test_df_copy['BMI'].apply(categorize_bmi).astype('category')

# Verify the new feature
train_df_copy[['BMI', 'BMI_Category']].head()
test_df_copy[['BMI', 'BMI_Category']].head()


,BMI,BMI_Category
0,25.75,Overweight
1,36.05,Obesity
2,29.18,Overweight
3,24.41,Normal weight
4,37.76,Obesity


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the latest DataFrame variables from the finished tasks
column_info_train = get_column_info(train_df_copy)
column_info_test = get_column_info(test_df_copy)

print("Train DataFrame Column Info:")
print(column_info_train)
print("\nTest DataFrame Column Info:")
print(column_info_test)


Train DataFrame Column Info:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': ['BMI_Category']}

Test DataFrame Column Info:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': ['BMI_Category']}


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from metagpt.tools.libs.data_preprocess import OneHotEncode

# One-hot encode categorical columns
categorical_features = ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History', 'BMI_Category']
one_hot_encoder = OneHotEncode(features=categorical_features)
train_df_encoded = one_hot_encoder.fit_transform(train_df_copy)
test_df_encoded = one_hot_encoder.transform(test_df_copy)

# Prepare the data for training
X_train = train_df_encoded.drop(columns=['Heart_Disease'])
y_train = train_df_encoded['Heart_Disease'].apply(lambda x: 1 if x == 'Yes' else 0)

X_test = test_df_encoded.drop(columns=['Heart_Disease'])
y_test = test_df_encoded['Heart_Disease'].apply(lambda x: 1 if x == 'Yes' else 0)

# Train the model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, max_depth=6, learning_rate=0.1)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Compute area under ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'Area under ROC curve: {roc_auc}')


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


KeyError: "['Heart_Disease'] not found in axis"

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from metagpt.tools.libs.data_preprocess import OneHotEncode

# One-hot encode categorical columns, excluding 'Heart_Disease'
categorical_features = ['General_Health', 'Checkup', 'Exercise', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History', 'BMI_Category']
one_hot_encoder = OneHotEncode(features=categorical_features)
train_df_encoded = one_hot_encoder.fit_transform(train_df_copy)
test_df_encoded = one_hot_encoder.transform(test_df_copy)

# Prepare the data for training
X_train = train_df_encoded.drop(columns=['Heart_Disease'])
y_train = train_df_encoded['Heart_Disease'].apply(lambda x: 1 if x == 'Yes' else 0)

X_test = test_df_encoded.drop(columns=['Heart_Disease'])
y_test = test_df_encoded['Heart_Disease'].apply(lambda x: 1 if x == 'Yes' else 0)

# Train the model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, max_depth=6, learning_rate=0.1)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Compute area under ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'Area under ROC curve: {roc_auc}')

D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:19:25] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Area under ROC curve: 0.8319314576023077
